# PixelPen: Handwritten Digit Recognition
### End-to-End Deep Learning Pipeline (EDA -> Augmentation -> CNN -> Evaluation -> Export)
Tagline: *"Draw it. Snap it. Let AI read it."*

This notebook trains and validates the Convolutional Neural Network (CNN) powering the PixelPen web application.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

## 1. Load and Inspect MNIST Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f"x_train shape: {x_train.shape}, dtype: {x_train.dtype}")
print(f"x_test shape: {x_test.shape}, dtype: {x_test.dtype}")
print(f"Labels distribution: {np.bincount(y_train)}")

# Normalize pixel intensities to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Expand dimension for single-channel grayscale (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

## 2. Visualize Sample MNIST Digits

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    axes[i].imshow(x_train[idx].squeeze(), cmap='gray')
    axes[i].set_title(f"Digit: {i}", fontsize=12, fontweight='bold')
    axes[i].axis('off')
plt.suptitle("Sample MNIST Digits (Clean Centered 28x28)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Augmentation Pipeline for Canvas & Photo Robustness

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.08, fill_mode="constant", fill_value=0.0),
    layers.RandomTranslation(0.1, 0.1, fill_mode="constant", fill_value=0.0),
    layers.RandomZoom(0.1, fill_mode="constant", fill_value=0.0),
    layers.RandomContrast(0.1)
], name="pixelpen_augmentation")

# Visualize augmented variants of a sample digit
sample_digit = tf.expand_dims(x_train[7], 0)
plt.figure(figsize=(10, 2.5))
for i in range(5):
    augmented_img = data_augmentation(sample_digit, training=True)
    plt.subplot(1, 5, i + 1)
    plt.imshow(augmented_img[0].numpy().squeeze(), cmap='inferno')
    plt.title(f"Augmented #{i+1}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Build the PixelPen CNN Model

In [ ]:
def build_model():
    inputs = keras.Input(shape=(28, 28, 1), name="digit_input")
    x = data_augmentation(inputs)
    
    # Conv Block 1
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Conv Block 2
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Dense Classification Block
    x = layers.Flatten()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax", name="probabilities")(x)
    
    return keras.Model(inputs=inputs, outputs=outputs, name="PixelPen_CNN")

model = build_model()
model.summary()

## 5. Compile & Train Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)
]

history = model.fit(
    x_train, y_train,
    epochs=12,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## 6. Evaluate on Test Set & Error Analysis

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

preds = model.predict(x_test, batch_size=256)
y_pred = np.argmax(preds, axis=1)

print(classification_report(y_test, y_pred, target_names=[f"Digit {i}" for i in range(10)]))

## 7. Save Model for FastAPI Backend Deployment

In [ ]:
os.makedirs("../backend/model", exist_ok=True)
model.save("../backend/model/digit_cnn.keras")
print("Model saved to ../backend/model/digit_cnn.keras")